# Hybrid RAG với Gemini & BM25

Notebook này trình bày cách kết hợp hai phương pháp truy xuất thông tin:
- **Semantic Search** (Gemini Embeddings + ChromaDB)
- **Keyword Search** (BM25)

Pipeline sử dụng Ensemble để lấy kết quả tốt nhất từ cả hai phương pháp.

## Các bước chính:
1. Cài đặt thư viện cần thiết
2. Thiết lập API Key cho Gemini
3. Tạo Embeddings và load dữ liệu
4. Tạo vectorstore và keyword retriever
5. Kết hợp hai phương pháp bằng EnsembleRetriever
6. Tạo LLM Gemini 2.5 Flash
7. Thiết lập RAG pipeline với prompt ngắn gọn
8. Kiểm tra kết quả và tạo dataset

In [1]:
! pip install --quiet chromadb rank_bm25 ragas langchain-google-genai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 2.9 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.7/20.7 MB 90.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 317.6/317.6 kB 20.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.8/57.8 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 56.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 73.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 467.1/467.1 kB 27.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.3/103.3 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.4/17.4 MB 99.3 MB/s eta 0:

In [6]:
!pip install -U langchain langchain-community langchain-core chromadb rank-bm25

In [2]:
import os
from google.colab import userdata

# Rất quan trọng: Sử dụng GOOGLE_API_KEY để tương thích tối đa
os.environ["GOOGLE_API_KEY"] = userdata.get('GEMINI_API_KEY')
print("GOOGLE_API_KEY đã được thiết lập thành công.")

GOOGLE_API_KEY đã được thiết lập thành công.


In [3]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
import os

embeddings = GoogleGenerativeAIEmbeddings(
    model="text-embedding-004",
    api_key=os.getenv("GEMINI_API_KEY") # Lấy Key từ biến môi trường
)

print("Gemini Embeddings đã được tải thành công.")

Gemini Embeddings đã được tải thành công.


---

## Semantic Search (Gemini Embeddings + ChromaDB)

- Sử dụng Gemini để tạo embeddings cho dữ liệu.
- Lưu embeddings vào ChromaDB để thực hiện truy vấn ngữ nghĩa.
- Phù hợp với các truy vấn cần hiểu ý nghĩa sâu xa của câu hỏi.


In [4]:
# load data
from langchain_community.document_loaders import CSVLoader
loader = CSVLoader("/content/sign_vocabulary.csv")
documents = loader.load()

In [5]:
# split documents
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(chunk_size=512, chunk_overlap=0)
documents = text_splitter.split_documents(documents)

In [7]:
from langchain_community.vectorstores import Chroma
vectorstore = Chroma.from_documents(documents, embeddings)

In [8]:
# create sematic search
retriever = vectorstore.as_retriever()

In [9]:
query = "Từ nào đặt tay trái trước ngực, rồi xoay tay phải xong đặt lên tay trái"
results = retriever.invoke(query)

for i, doc in enumerate(results, 1):
    print(f"--- Kết quả {i} ---")
    print(doc.page_content)

--- Kết quả 1 ---
﻿unit_id: 2
type: từ vựng
text: Tuổi
description: Tay trái nắm, lòng bàn tay hướng vào trong, đặt trước ngực. Tay phải khép, hơi khum lại, lòng bàn tay hướng ra ngoài, đầu ngón tay hướng chếch lên trên, đặt trên tay trái, giữ nguyên cánh tay, xoay cổ tay một vòng từ ngoài vào trong đồng thời nắm tay lại rồi chạm vào tay trái.
--- Kết quả 2 ---
﻿unit_id: 1
type: từ vựng
text: Chào
description: Tay phải khép, lòng bàn tay hướng về phía trước, đầu ngón tay hướng lên trên, đặt bên phải, ngang tầm với mặt, đưa nhẹ tay sang hai bên (lặp lại 1–2 lần).
--- Kết quả 3 ---
﻿unit_id: 3
type: từ vựng
text: Xin lỗi
description: Tay trái khép, lòng bàn tay ngửa, đầu ngón tay hướng chếch sang phải, đặt trước ngực. Tay phải khép, lòng bàn tay úp, đầu ngón tay hướng chếch sang trái, đặt các đầu ngón tay vào lòng bàn tay trái, đưa tay dọc trong lòng bàn tay trái (lặp lại 2–3 lần).
--- Kết quả 4 ---
﻿unit_id: 4
type: từ vựng
text: Màu sắc
description: 1. Tay phải giống chữ cái ngón tay “

---

## Keyword Search (BM25)

- Sử dụng thuật toán BM25 để truy vấn dựa trên từ khóa.
- Phù hợp với các truy vấn có từ khóa rõ ràng, không cần hiểu ngữ nghĩa sâu.
- Trả về các kết quả dựa trên mức độ khớp từ khóa với dữ liệu.


In [10]:
# create keyword retriever
from langchain_community.retrievers import BM25Retriever
keyword_retriever = BM25Retriever.from_documents(documents)
keyword_retriever.k =  3
query = "Từ nào đặt tay trái trước ngực, rồi xoay tay phải xong đặt lên tay trái"
results = keyword_retriever.invoke(query)

for i, doc in enumerate(results, 1):
    print(f"--- Kết quả {i} ---")
    print(doc.page_content)

--- Kết quả 1 ---
﻿unit_id: 2
type: từ vựng
text: Tuổi
description: Tay trái nắm, lòng bàn tay hướng vào trong, đặt trước ngực. Tay phải khép, hơi khum lại, lòng bàn tay hướng ra ngoài, đầu ngón tay hướng chếch lên trên, đặt trên tay trái, giữ nguyên cánh tay, xoay cổ tay một vòng từ ngoài vào trong đồng thời nắm tay lại rồi chạm vào tay trái.
--- Kết quả 2 ---
﻿unit_id: 3
type: từ vựng
text: Xin lỗi
description: Tay trái khép, lòng bàn tay ngửa, đầu ngón tay hướng chếch sang phải, đặt trước ngực. Tay phải khép, lòng bàn tay úp, đầu ngón tay hướng chếch sang trái, đặt các đầu ngón tay vào lòng bàn tay trái, đưa tay dọc trong lòng bàn tay trái (lặp lại 2–3 lần).
--- Kết quả 3 ---
﻿unit_id: 8
type: từ vựng
text: Màu nâu
description: 1. Tay phải giống chữ cái ngón tay “M”, đặt trước ngực, giữ nguyên cánh tay, lắc cổ tay về bên phải. 2. Tay phải giống chữ cái ngón tay “N”, đặt trước ngực, lắc cổ tay sang phải.


In [ ]:
!pip install -U langchain==0.2.11 langchain-community


In [18]:
!pip install -q langchain

---

## Hybrid Search (EnsembleRetriever)

- Kết hợp kết quả từ cả hai phương pháp: Semantic Search và Keyword Search.
- Tính điểm cho từng kết quả dựa trên thứ hạng và trọng số của từng phương pháp.
- Trả về các kết quả tổng hợp, giúp tăng độ chính xác và đa dạng cho truy vấn.


In [23]:
from langchain_core.documents import Document
from collections import defaultdict

class EnsembleRetriever:
    def __init__(self, vectorstore, keyword_retriever, weights=None):
        self.vectorstore = vectorstore
        self.keyword_retriever = keyword_retriever
        self.weights = weights or [1.0, 1.0]

    def vector_retriever(self, query, k=3):
        results = self.vectorstore.similarity_search(query, k=k)
        return [Document(page_content=d.page_content, metadata=d.metadata) for d in results]

    def bm25_retriever(self, query, k=3):
        results = self.keyword_retriever.invoke(query)
        return [Document(page_content=d.page_content, metadata=d.metadata) for d in results]

    def get_relevant_documents(self, query, top_k=10):
        retrievers = [self.vector_retriever, self.bm25_retriever]
        scores = defaultdict(float)
        doc_map = {}

        for retriever, weight in zip(retrievers, self.weights):
            docs = retriever(query)
            for rank, doc in enumerate(docs):
                scores[doc.page_content] += weight / (rank + 1)
                doc_map[doc.page_content] = doc

        top_docs = sorted(doc_map.values(), key=lambda d: scores[d.page_content], reverse=True)
        return top_docs[:top_k]

In [26]:
# Test
ensemble_retriever = EnsembleRetriever(vectorstore, keyword_retriever, weights=[0.5, 0.5])
docs = ensemble_retriever.get_relevant_documents(
    "Từ nào đặt tay trái trước ngực, rồi xoay tay phải xong đặt lên tay trái",
    top_k=3
)
for i, d in enumerate(docs, 1):
    print(f"--- Kết quả {i} ---")
    print(d.page_content)


--- Kết quả 1 ---
﻿unit_id: 2
type: từ vựng
text: Tuổi
description: Tay trái nắm, lòng bàn tay hướng vào trong, đặt trước ngực. Tay phải khép, hơi khum lại, lòng bàn tay hướng ra ngoài, đầu ngón tay hướng chếch lên trên, đặt trên tay trái, giữ nguyên cánh tay, xoay cổ tay một vòng từ ngoài vào trong đồng thời nắm tay lại rồi chạm vào tay trái.
--- Kết quả 2 ---
﻿unit_id: 3
type: từ vựng
text: Xin lỗi
description: Tay trái khép, lòng bàn tay ngửa, đầu ngón tay hướng chếch sang phải, đặt trước ngực. Tay phải khép, lòng bàn tay úp, đầu ngón tay hướng chếch sang trái, đặt các đầu ngón tay vào lòng bàn tay trái, đưa tay dọc trong lòng bàn tay trái (lặp lại 2–3 lần).
--- Kết quả 3 ---
﻿unit_id: 1
type: từ vựng
text: Chào
description: Tay phải khép, lòng bàn tay hướng về phía trước, đầu ngón tay hướng lên trên, đặt bên phải, ngang tầm với mặt, đưa nhẹ tay sang hai bên (lặp lại 1–2 lần).


In [62]:
from langchain_google_genai import ChatGoogleGenerativeAI
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.1)
print("Gemini 2.5 Flash LLM đã được khởi tạo.")

Gemini 2.5 Flash LLM đã được khởi tạo.


In [29]:
!pip install --upgrade langchain

In [61]:
# =========================
# RAG Chain
# =========================
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

# --- 1. Prompt trích xuất ---
prompt_extract = ChatPromptTemplate.from_messages([
    ("system",
     "Chỉ trả lời theo định dạng: "
     "'Với hành động {{input}} là từ {{content}}'. "
     "Nếu không tìm thấy từ phù hợp, trả lời 'Không biết'. "
     "{{content}} phải xuất hiện NGUYÊN VĂN trong Context."),
    ("human",
     "Context:\n{context}\n\nQuestion: {input}")
])

# --- Format context ---
def format_docs(docs):
    return "\n\n---\n\n".join(doc.page_content for doc in docs)

# --- Bọc retriever ---
def retrieve_documents(query):
    return ensemble_retriever.get_relevant_documents(query)

retrieval_runnable = RunnableLambda(retrieve_documents)

# --- Ràng buộc LLM ---
llm_strict = llm.bind(stop=["\n"], temperature=0)  # Dùng cho trích xuất

# --- Pipeline trích xuất ---
rag_extract = (
    {"context": retrieval_runnable | RunnableLambda(format_docs), "input": RunnablePassthrough()}
    | prompt_extract
    | llm_strict
    | StrOutputParser()
)

print("✅ RAG chain trích xuất đã được thiết lập thành công.")

# --- Kiểm thử ---
test_query = "Từ nào đặt tay trái trước ngực, rồi xoay tay phải xong đặt lên tay trái"
print("Trích xuất:", rag_extract.invoke(test_query))


✅ RAG chain trích xuất đã được thiết lập thành công.
Trích xuất: Với hành động đặt tay trái trước ngực, rồi xoay tay phải xong đặt lên tay trái là từ Tuổi


In [64]:
# create dataset
questions = ["Từ nào đặt tay trái trước ngực, rồi xoay tay phải xong đặt lên tay trái", "Từ nào phải làm 2 tay giống số 1"]
response = []
contexts = []

# Inference
for query in questions:
  response.append(rag_extract.invoke(query))
  contexts.append([docs.page_content for docs in ensemble_retriever.get_relevant_documents(query)])

# To dict
data = {
    "query": questions,
    "response": response,
    "context": contexts,
}

In [65]:
# create dataset
from datasets import Dataset
dataset = Dataset.from_dict(data)

In [66]:
# create dataframe
import pandas as pd
df = pd.DataFrame(dataset)

In [60]:
df

,query,response,context
0,"Từ nào đặt tay trái trước ngực, rồi xoay tay p...","Với hành động đặt tay trái trước ngực, rồi xoa...",[﻿unit_id: 2\ntype: từ vựng\ntext: Tuổi\ndescr...
1,Từ nào phải làm 2 tay giống số 1,Với hành động làm 2 tay giống số 1 là từ Màu sắc.,[﻿unit_id: 1\ntype: từ vựng\ntext: Chào\ndescr...


Error: Runtime no longer has a reference to this dataframe, please re-run this cell and try again.
